# Daten laden und erste Exploration

In diesem Notebook wird der Datensatz geladen und zunächst untersucht. Ziel ist es, einen Überblick über die Struktur der Daten, die vorhandenen Spalten sowie mögliche fehlende Werte zu erhalten.

In [45]:
import pandas as pd

## Datensatz laden

Der Datensatz wird mit der Bibliothek pandas in die Python-Umgebung geladen. Anschließend werden die ersten Zeilen angezeigt, um einen ersten Eindruck der Daten zu erhalten.

In [46]:
df = pd.read_csv("../data/consumer_complaints_sample.csv")

In [47]:
df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,2020-05-08,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,These are not my accounts.,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,NV,89030,NaN,Consent provided,Web,2020-05-08,Closed with explanation,Yes,NaN,3642453
1,2024-01-05,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,Kindly address this issue on my credit report....,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,IL,60502,NaN,Consent provided,Web,2024-01-05,Closed with non-monetary relief,Yes,NaN,8113747
2,2020-03-19,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,"I wrote three requests, the unverified account...",NaN,"EQUIFAX, INC.",NC,28562,NaN,Consent provided,Web,2020-03-19,Closed with explanation,Yes,NaN,3573294
3,2019-10-22,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Old information reappears or never goes away,XXXX XXXX has a old account settled in XXXX th...,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,HI,967XX,Servicemember,Consent provided,Web,2019-10-22,Closed with non-monetary relief,Yes,NaN,3414709
4,2020-03-29,Student loan,Federal student loan servicing,Dealing with your lender or servicer,Keep getting calls about your loan,They call at all hours and on the weekends usi...,NaN,"Navient Solutions, LLC.",CA,92028,Servicemember,Consent provided,Web,2020-03-29,Closed with explanation,Yes,NaN,3584679


## Überblick über die Datenstruktur

Mit `df.info()` wird untersucht:
- wie viele Zeilen und Spalten vorhanden sind,
- welche Datentypen verwendet werden,
- und ob fehlende Werte existieren.

Dies ist wichtig, um die Datenqualität vor der Analyse zu prüfen.

In [48]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column                        Non-Null Count  Dtype
---  ------                        --------------  -----
 0   Date received                 5000 non-null   str  
 1   Product                       5000 non-null   str  
 2   Sub-product                   4934 non-null   str  
 3   Issue                         5000 non-null   str  
 4   Sub-issue                     4563 non-null   str  
 5   Consumer complaint narrative  5000 non-null   str  
 6   Company public response       2720 non-null   str  
 7   Company                       5000 non-null   str  
 8   State                         4981 non-null   str  
 9   ZIP code                      5000 non-null   str  
 10  Tags                          479 non-null    str  
 11  Consumer consent provided?    5000 non-null   str  
 12  Submitted via                 5000 non-null   str  
 13  Date sent to company          5000 non-null 

## Analyse fehlender Werte

In diesem Schritt wird überprüft, welche Spalten fehlende Werte enthalten. Fehlende Werte können die spätere Analyse beeinflussen und müssen daher identifiziert werden.

In [49]:
df.isnull().sum()

Date received                      0
Product                            0
Sub-product                       66
Issue                              0
Sub-issue                        437
Consumer complaint narrative       0
Company public response         2280
Company                            0
State                             19
ZIP code                           0
Tags                            4521
Consumer consent provided?         0
Submitted via                      0
Date sent to company               0
Company response to consumer       0
Timely response?                   0
Consumer disputed?              4803
Complaint ID                       0
dtype: int64

## Untersuchung der vorhandenen Spalten

Zur Vorbereitung der NLP-Analyse werden alle Spaltennamen angezeigt. Ziel ist es, diejenige Spalte zu identifizieren, die die eigentlichen Beschwerdetexte enthält.

In [50]:
df.columns

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID'],
      dtype='str')

## Identifikation der relevanten Textspalte

Die Spalte `Consumer complaint narrative` enthält die eigentlichen freien Beschwerdetexte der Verbraucher:innen. Dies lässt sich daran erkennen, dass die Inhalte vollständige Sätze und längere Beschreibungen von Problemen enthalten.

Diese Spalte ist für die NLP-Analyse besonders wichtig, da:
- sie unstrukturierte Textdaten enthält,
- die Beschwerden in natürlicher Sprache formuliert sind,
- und daraus Themen, Muster und häufige Probleme extrahiert werden können.

Die weiteren Schritte der Textanalyse basieren daher hauptsächlich auf dieser Spalte.

In [51]:
df['Consumer complaint narrative'].head()

0                           These are not my accounts.
1    Kindly address this issue on my credit report....
2    I wrote three requests, the unverified account...
3    XXXX XXXX has a old account settled in XXXX th...
4    They call at all hours and on the weekends usi...
Name: Consumer complaint narrative, dtype: str

## Erste Beobachtungen

Die ersten Beispiele zeigen, dass die Beschwerden:
- unterschiedlich lang sind,
- viele Sonderzeichen und Formatierungen enthalten,
- und sprachlich nicht standardisiert sind.

Daher ist eine Vorverarbeitung notwendig, bevor die Texte mit NLP-Methoden analysiert werden können.

## Entfernen fehlender Beschwerdetexte

Einige Einträge enthalten keine eigentlichen Beschwerdetexte und werden als `NaN` angezeigt. Diese Datensätze können nicht für die NLP-Analyse verwendet werden und müssen daher entfernt werden.

In [52]:
df = df.dropna(subset=['Consumer complaint narrative'])

In [53]:
df['Consumer complaint narrative'].isnull().sum()

np.int64(0)

## Vereinheitlichung der Spaltennamen

Zur Vereinfachung der weiteren Analyse werden die Spaltennamen vereinheitlicht. Dabei werden Leerzeichen durch Unterstriche ersetzt und alle Spaltennamen in Kleinbuchstaben umgewandelt.

In [54]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

In [55]:
df.columns

Index(['date_received', 'product', 'sub-product', 'issue', 'sub-issue',
       'consumer_complaint_narrative', 'company_public_response', 'company',
       'state', 'zip_code', 'tags', 'consumer_consent_provided?',
       'submitted_via', 'date_sent_to_company', 'company_response_to_consumer',
       'timely_response?', 'consumer_disputed?', 'complaint_id'],
      dtype='str')

In [56]:
df['consumer_complaint_narrative']

0                              These are not my accounts.
1       Kindly address this issue on my credit report....
2       I wrote three requests, the unverified account...
3       XXXX XXXX has a old account settled in XXXX th...
4       They call at all hours and on the weekends usi...
                              ...                        
4995    I have submitted two credit report disputes wi...
4996    ALL REPORTED DATA MUST BE COMPLETE, CORRECT, A...
4997    On XX/XX/XXXX I logged onto XXXX XXXX to check...
4998    I authorized a {$100.00} charge on a recorded ...
4999    On XX/XX/XXXX, I went to a Santander branch to...
Name: consumer_complaint_narrative, Length: 5000, dtype: str

## Zusammenfassung der ersten Datenanalyse

Der Datensatz wurde erfolgreich geladen und untersucht. Die Spalte `consumer_complaint_narrative` wurde als wichtigste Textquelle für die NLP-Analyse identifiziert. Zusätzlich wurden fehlende Werte entfernt und die Spaltennamen vereinheitlicht, um die weitere Verarbeitung zu erleichtern.

Damit sind die Daten für die Vorverarbeitung vorbereitet.

## Speicherung des bereinigten Datensatzes

Der vorbereitete Datensatz wird gespeichert, damit die folgenden Verarbeitungsschritte ohne erneutes Laden und Bereinigen der Rohdaten durchgeführt werden können.

In [57]:
df.to_csv("../processed_data/01_loaded_data.csv", index=False)